In [15]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline




In [16]:
london = pd.read_csv("data/london.csv")
paris = pd.read_csv("data/paris.csv")
amsterdam = pd.read_csv("data/amsterdam.csv")
new_york = pd.read_csv("data/newyork.csv")


In [17]:
# did,t used merge as merge is used for stacking on basis of column name
# pd.merge(london,paris,amsterdam,new_york,on="city")
#we will use concat
#ignore_index=True tells Pandas to create a new continuous index after concatenating.
df = pd.concat(
    [london, paris, amsterdam, new_york],
    ignore_index=True
)

In [18]:
df.isnull().sum()
df = df[(df["price"] >= 0) & (df["price"] <= 600)]


In [19]:
X = df[[
    "city",
    "neighbourhood_cleansed",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "longitude",
    "latitude"
    
]]

In [20]:
# df["log_price"] = np.log1p(df["price"])
# y = df["log_price"]
y=df["price"]

In [21]:
X.info()
y.info()

<class 'pandas.core.frame.DataFrame'>
Index: 127643 entries, 0 to 138217
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   city                    127643 non-null  object 
 1   neighbourhood_cleansed  127643 non-null  object 
 2   room_type               127643 non-null  object 
 3   accommodates            127643 non-null  int64  
 4   bathrooms               127643 non-null  float64
 5   bedrooms                127643 non-null  float64
 6   beds                    127643 non-null  float64
 7   longitude               127643 non-null  float64
 8   latitude                127643 non-null  float64
dtypes: float64(5), int64(1), object(3)
memory usage: 9.7+ MB
<class 'pandas.core.series.Series'>
Index: 127643 entries, 0 to 138217
Series name: price
Non-Null Count   Dtype  
--------------   -----  
127643 non-null  float64
dtypes: float64(1)
memory usage: 1.9 MB


In [22]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [23]:
# syntax
# preprocessor = ColumnTransformer(
#     transformers=[
#         (
#             "name",
#             transformer,
#             columns
#         ),
#         (
#             "name2",
#             transformer2,
#             columns2
#         )
#     ],
#     remainder="drop"   # optional
# )

#OneHotEncoder
# handle_unknown="ignore"
# This prevents errors if the model encounters a category during prediction that wasn't present during training.

preprocessor=ColumnTransformer(
                transformers=[
                    ("num",StandardScaler(),["bathrooms","bedrooms","beds","accommodates","longitude","latitude"]),
                    ("cat",OneHotEncoder(handle_unknown="ignore"),["city","neighbourhood_cleansed","room_type"])
                             ])


In [24]:
pipeline=Pipeline([
    ("preprocessing",preprocessor),
    ("lr",LinearRegression())
])


In [25]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bathrooms', 'bedrooms',
                                                   'beds', 'accommodates',
                                                   'longitude', 'latitude']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city',
                                                   'neighbourhood_cleansed',
                                                   'room_type'])])),
                ('lr', LinearRegression())])

In [26]:
y_pred = pipeline.predict(X_test)

# print("R² (log scale):", r2_score(y_test, y_pred_log))

# y_pred = np.expm1(y_pred_log)
# y_actual = np.expm1(y_test)

print("R² (price scale):", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R² (price scale): 0.4483560183126244
MAE: 69.65276186645153
RMSE: 95.15642225782092


In [27]:
import pandas as pd

test = pd.DataFrame([
    {
        "city": "Paris",
        "neighbourhood_cleansed": "Vaugirard",
        "room_type": "Entire home/apt",
        "accommodates": 2,
        "bathrooms": 1.0,
        "bedrooms": 1.0,
        "beds": 1.0,
        "longitude": 2.28936,
        "latitude": 48.8495
    },
    {
        "city": "Paris",
        "neighbourhood_cleansed": "Buttes-Montmartre",
        "room_type": "Entire home/apt",
        "accommodates": 4,
        "bathrooms": 1.0,
        "bedrooms": 2.0,
        "beds": 2.0,
        "longitude": 2.36451,
        "latitude": 48.89187
    },
    {
        "city": "Paris",
        "neighbourhood_cleansed": "Gobelins",
        "room_type": "Entire home/apt",
        "accommodates": 2,
        "bathrooms": 1.0,
        "bedrooms": 1.0,
        "beds": 1.0,
        "longitude": 2.35404,
        "latitude": 48.82971
    },
    {
        "city": "Paris",
        "neighbourhood_cleansed": "Entrepôt",
        "room_type": "Entire home/apt",
        "accommodates": 1,
        "bathrooms": 1.0,
        "bedrooms": 1.0,
        "beds": 1.0,
        "longitude": 2.36478,
        "latitude": 48.87278
    }
])

actual_prices = [226.75, 225.00, 95.33, 37.40]

predicted_prices = pipeline.predict(test)

for i in range(len(test)):
    print(f"Listing {i+1}")
    print(f"Actual Price    : ${actual_prices[i]:.2f}")
    print(f"Predicted Price : ${predicted_prices[i]:.2f}")
    print(f"Absolute Error  : ${abs(predicted_prices[i]-actual_prices[i]):.2f}")
    print("-"*40)

Listing 1
Actual Price    : $226.75
Predicted Price : $184.97
Absolute Error  : $41.78
----------------------------------------
Listing 2
Actual Price    : $225.00
Predicted Price : $234.26
Absolute Error  : $9.26
----------------------------------------
Listing 3
Actual Price    : $95.33
Predicted Price : $148.12
Absolute Error  : $52.79
----------------------------------------
Listing 4
Actual Price    : $37.40
Predicted Price : $164.22
Absolute Error  : $126.82
----------------------------------------


In [28]:
from joblib import dump

dump(pipeline, "models/price_prediction_model.joblib")

['models/price_prediction_model.joblib']